# Prior sensitivity analysis

This notebook inspects the outputs from `prior_sensitivity_analysis.py`. It checks bound violations, compares sampled distributions with the declared uniform priors, and exposes correlations or edge effects that could influence the large simulations.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULT_DIR = Path('results/prior_sensitivity')
metadata = json.loads((RESULT_DIR / 'metadata.json').read_text())
metadata['n_samples'], metadata['models'].keys()

In [ ]:
rows = []
for model, result in metadata['models'].items():
    for parameter, summary in result['summary'].items():
        rows.append({'model': model, 'parameter': parameter, **summary})
summary_table = pd.DataFrame(rows)
summary_table[['model', 'parameter', 'declared_lower', 'declared_upper', 'sample_min', 'sample_max', 'sample_mean', 'sample_std', 'below_bound', 'above_bound']]

In [ ]:
violations = []
for model, result in metadata['models'].items():
    violations.extend({'model': model, **item} for item in result['violations'])

violation_table = pd.DataFrame(violations)
print('Recorded violations:', len(violation_table))
violation_table.head()

In [ ]:
for model, result in metadata['models'].items():
    arrays = np.load(RESULT_DIR / f'{model}.npz')
    parameters = list(arrays.files)
    fig, axes = plt.subplots(len(parameters), 1, figsize=(9, 3 * len(parameters)))
    axes = np.atleast_1d(axes)
    for axis, parameter in zip(axes, parameters):
        values = arrays[parameter]
        bounds = result['summary'][parameter]
        axis.hist(values, bins=100, density=True, alpha=0.75)
        axis.axvline(bounds['declared_lower'], color='black', linestyle='--')
        axis.axvline(bounds['declared_upper'], color='black', linestyle='--')
        axis.set_title(f'{model}: {parameter}')
        axis.set_ylabel('density')
    axes[-1].set_xlabel('sampled value')
    fig.tight_layout()
    plt.show()

In [ ]:
for model in metadata['models']:
    arrays = np.load(RESULT_DIR / f'{model}.npz')
    frame = pd.DataFrame({parameter: arrays[parameter] for parameter in arrays.files})
    correlation = frame.corr(numeric_only=True)
    print(model)
    display(correlation.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1))

## Interpretation

A nonzero violation count indicates a sampler or bounds mismatch. Large deviations between the sample mean and the midpoint indicate a non-uniform sampler or a parameter transformation. Strong correlations are unexpected for the current independent-draw implementation and would warrant checking the random-number handling in the simulation workers.